In [0]:
display(dbutils.fs.ls("/Volumes/fia_databricks/datos/fia_volume/"))

In [0]:

from pyspark.sql import functions as F

# Ruta del archivo original en S3
ruta_raw = (
    "s3://fia-renipress-databricks-2026/"
    "renipress/RENIPRESS_31-08-2026.csv"
)

# Leer el CSV respetando el separador y las comillas
df_raw = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("sep", ";")
    .option("encoding", "UTF-8")
    .option("inferSchema", "false")
    .option("quote", '"')
    .option("multiLine", "true")
    .option("mode", "PERMISSIVE")
    .load(ruta_raw)
)

print("LECTURA RAW COMPLETADA")
print("Total de registros:", df_raw.count())
print("Total de columnas:", len(df_raw.columns))

display(df_raw.limit(10))


In [0]:

# Verificar las columnas detectadas
print("Número de columnas:", len(df_raw.columns))

for numero, columna in enumerate(df_raw.columns, start=1):
    print(f"{numero}. {columna}")

# Revisar el registro del hospital de código 00000308
display(
    df_raw
    .filter(F.col("COD_IPRESS") == "00000308")
    .select(
        "COD_IPRESS",
        "NOMBRE",
        "DEPARTAMENTO",
        "PROVINCIA",
        "DISTRITO",
        "CATEGORIA",
        "ESTADO"
    )
)

# Buscar departamentos con el valor sospechoso ACTIVO
display(
    df_raw.filter(
        F.upper(F.trim(F.col("DEPARTAMENTO"))) == "ACTIVO"
    )
)


In [0]:

# Crear SILVER a partir del RAW
df_silver = df_raw

# Limpiar espacios y convertir cadenas vacías en NULL
for columna in df_silver.columns:
    df_silver = df_silver.withColumn(
        columna,
        F.when(
            F.trim(F.col(columna)) == "",
            F.lit(None).cast("string")
        ).otherwise(
            F.trim(F.col(columna))
        )
    )

# Convertir la fecha de inicio de actividad.
# Las cadenas que no sean fechas válidas quedarán como NULL.
df_silver = df_silver.withColumn(
    "INICIO_ACTIVIDAD",
    F.expr("try_to_timestamp(INICIO_ACTIVIDAD)")
)

# Convertir coordenadas a valores numéricos
df_silver = (
    df_silver
    .withColumn(
        "NORTE",
        F.expr("try_cast(NORTE AS DOUBLE)")
    )
    .withColumn(
        "ESTE",
        F.expr("try_cast(ESTE AS DOUBLE)")
    )
)

print("LIMPIEZA SILVER COMPLETADA")
print("Registros:", df_silver.count())
print("Columnas:", len(df_silver.columns))

display(
    df_silver.select(
        "COD_IPRESS",
        "NOMBRE",
        "DEPARTAMENTO",
        "PROVINCIA",
        "DISTRITO",
        "CATEGORIA",
        "INICIO_ACTIVIDAD",
        "NORTE",
        "ESTE"
    ).limit(10)
)


In [0]:

# Resumen de calidad de SILVER
total_registros = df_silver.count()

print("Total de registros:", total_registros)
print("Total de columnas:", len(df_silver.columns))

# Contar valores vacíos en campos importantes
campos = [
    "COD_IPRESS",
    "NOMBRE",
    "DEPARTAMENTO",
    "CATEGORIA"
]

for campo in campos:
    cantidad_nulos = df_silver.filter(
        F.col(campo).isNull() |
        (F.trim(F.col(campo)) == "")
    ).count()

    print(f"{campo}: {cantidad_nulos} valores vacíos")

# Buscar códigos IPRESS duplicados
duplicados = (
    df_silver
    .filter(F.col("COD_IPRESS").isNotNull())
    .groupBy("COD_IPRESS")
    .count()
    .filter(F.col("count") > 1)
)

print("Cantidad de códigos IPRESS duplicados:", duplicados.count())

display(duplicados.orderBy(F.desc("count")))

# Revisar departamentos
display(
    df_silver
    .groupBy("DEPARTAMENTO")
    .count()
    .orderBy(F.desc("count"))
)

# Revisar categorías
display(
    df_silver
    .groupBy("CATEGORIA")
    .count()
    .orderBy(F.desc("count"))
)


In [0]:

nombre_tabla = "fia_databricks.datos.renipress_silver"

(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(nombre_tabla)
)

print("TABLA SILVER GUARDADA CORRECTAMENTE")
print("Tabla:", nombre_tabla)
